In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
import sys
from pathlib import Path

# Go to project root (REALTIMEFRAUDDETECTION)
project_root = Path().resolve().parent
sys.path.append(str(project_root))

In [4]:
df=pd.read_csv("../data/dataset_PFE_CDM_complet.csv")
df.shape

(1100000, 17)

In [5]:
df['Time']=pd.to_datetime(df['Time'])
df['Age']=df['Age'].astype(int)

df['Hour']=df['Time'].dt.hour
df['Date']=df['Time'].dt.date

In [6]:
df.sort_values(['Account Number','Time'],inplace=True)
df['rank']=df.groupby("Account Number")['Time'].rank(method='first',pct=True)

# Feature Engineering

In [7]:
from src.features.TimeFeatures import ComputeTimeFeatures
from src.features.CatEntropy import ComputeCatEntropy
from src.features.CatFreq import ComputeCatFreq
from src.preprocessing.TrainTestSplit import split

In [8]:
df=ComputeTimeFeatures(df)
df=ComputeCatEntropy(df)
df=ComputeCatFreq(df)

In [9]:
df=df[['Age','rank',
       'LogAmount', 'MovingAvg','MovingStd', 'LogTimeDiff','Hour',
       'TransactionTypeEntropy', 'ChannelEntropy','CardTypeEntropy', 'MerchandEntropy', 'CountryEntropy', 'CityEntropy',
       'TransactionTypeFreq','ChannelFreq','CardTypeFreq', 'MerchandFreq', 'CountryFreq','CityFreq']]

traindf,testdf=split(df)

In [15]:
traindf.shape

(879198, 18)

# Isolation Forest

In [10]:
from src.models.IsolationForest import IsolationForestModel
from src.training.train import trainmodel

2026-03-03 15:08:56.363804: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-03 15:08:56.414877: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-03 15:08:57.826961: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [11]:
iso=IsolationForestModel()

trainmodel(iso,traindf,runname='testIF')

2026/03/03 15:10:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/03 15:10:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


{'score_mean': -0.0710953874055305,
 'score_std': 0.035725661787105185,
 'score_p95': -5.776976458511142e-17}

In [12]:
def ComputeAvgPathLength(model, X):
    depths = []

    for tree in model.estimators_:
        node_indicator = tree.decision_path(X)
        depth = node_indicator.sum(axis=1)
        depths.append(np.array(depth).reshape(-1))

    depths = np.array(depths)
    return depths.mean(axis=0)

# AutoEncoder

In [13]:
from src.models.AutoEncoder import AutoEncoderModel

In [14]:
inputdim=traindf.shape[1]

ae=AutoEncoderModel(inputdim=inputdim,epochs=20)

trainmodel(ae,traindf,runname='testAE')

2026-03-03 15:11:00.989331: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/20
13738/13738 ━━━━━━━━━━━━━━━━━━━━ 30s 2ms/step - loss: 0.2319
Epoch 2/20
13738/13738 ━━━━━━━━━━━━━━━━━━━━ 28s 2ms/step - loss: 0.0264
Epoch 3/20
13738/13738 ━━━━━━━━━━━━━━━━━━━━ 28s 2ms/step - loss: 0.0237
Epoch 4/20
13738/13738 ━━━━━━━━━━━━━━━━━━━━ 28s 2ms/step - loss: 0.0227
Epoch 5/20
13738/13738 ━━━━━━━━━━━━━━━━━━━━ 29s 2ms/step - loss: 0.0221
Epoch 6/20
13738/13738 ━━━━━━━━━━━━━━━━━━━━ 28s 2ms/step - loss: 0.0218
Epoch 7/20
13738/13738 ━━━━━━━━━━━━━━━━━━━━ 29s 2ms/step - loss: 0.0213
Epoch 8/20
13738/13738 ━━━━━━━━━━━━━━━━━━━━ 32s 2ms/step - loss: 0.0211
Epoch 9/20
13738/13738 ━━━━━━━━━━━━━━━━━━━━ 28s 2ms/step - loss: 0.0210
Epoch 10/20
13738/13738 ━━━━━━━━━━━━━━━━━━━━ 29s 2ms/step - loss: 0.0209
Epoch 11/20
13738/13738 ━━━━━━━━━━━━━━━━━━━━ 28s 2ms/step - loss: 0.0208
Epoch 12/20
13738/13738 ━━━━━━━━━━━━━━━━━━━━ 28s 2ms/step - loss: 0.0207
Epoch 13/20
13738/13738 ━━━━━━━━━━━━━━━━━━━━ 28s 2ms/step - loss: 0.0206
Epoch 14/20
13738/13738 ━━━━━━━━━━━━━━━━━━━━ 28s 2ms/step - 

2026/03/03 15:21:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/03 15:21:26 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.


{'score_mean': 0.01503034884228775,
 'score_std': 0.013749243997979664,
 'score_p95': 0.04709612723002877}

# SelfOrganizingMap

In [15]:
from src.models.SelfOrganizingMap import SOMModel

In [16]:
model=SOMModel(inputlen=inputdim)

trainmodel(model,traindf,runname='testSOM')

{'score_mean': 4.211144652869979,
 'score_std': 2.240520669929512,
 'score_p95': 8.703826860727398}

# HyperParam Tuning

In [10]:
from src.training.tune import tunemodel
from src.models.IsolationForest import IsolationForestModel
from src.training.seachspaces import ifsearchspace

2026-03-03 21:09:10.391143: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-03 21:09:10.438912: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-03 21:09:12.144072: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [11]:
study = tunemodel(
    IsolationForestModel,
    ifsearchspace,
    traindf,
    ntrials=30
)

[I 2026-03-03 21:09:14,402] A new study created in memory with name: no-name-ef7e5aef-33ce-4fac-8306-c0dba8470f9f
[I 2026-03-03 21:11:58,250] Trial 0 finished with value: -0.08474522178507381 and parameters: {'nestimators': 388, 'maxsamples': 0.6481067442940248, 'contamination': 0.03499229156871427}. Best is trial 0 with value: -0.08474522178507381.
[I 2026-03-03 21:12:50,327] Trial 1 finished with value: -0.08035410197344993 and parameters: {'nestimators': 137, 'maxsamples': 0.504414845639024, 'contamination': 0.04082508285043356}. Best is trial 0 with value: -0.08474522178507381.
[I 2026-03-03 21:14:56,410] Trial 2 finished with value: -0.04733702138619137 and parameters: {'nestimators': 346, 'maxsamples': 0.5302152734452978, 'contamination': 0.0983388024090433}. Best is trial 0 with value: -0.08474522178507381.
[I 2026-03-03 21:17:59,686] Trial 3 finished with value: -0.06383504123197137 and parameters: {'nestimators': 370, 'maxsamples': 0.9001745129559438, 'contamination': 0.060457

In [15]:
print(study.best_params)

{'nestimators': 220, 'maxsamples': 0.6599937373902977, 'contamination': 0.010491751535066466}
